# Tower of Hanoi

This notebook is a playground to prototype the Tower of Hanoi before implementing it in Rust.

In [ ]:
'''
First we have to define the variables in place.

Columns are always three, we could use an enumerator or simply an index.
Probably with the latter we could make the algorithm eventually capable of working with any number of columns.
But quite frankly, with more columns the game becomes increasingly simpler, therefore we should limit to the default three.
'''

from enum import IntEnum

class Column(IntEnum):
    ORIGIN = 0
    MID = 1
    DESTINATION = 2

# The number of rings is definitely variable. At least 1 (otherwise the game cannot be played!), virtually up to infinity.
NUM_RINGS: int = 3
assert NUM_RINGS > 0, "You need at least one ring to play, duh!!"

# For now, that's it, let's not overcomplicate it initially. Might add additional constraints later

'''
The initial state of a Tower of Hanoi always has the whole tower, correctly placed, on the source column.
Therefore, no matter the number of rings, they are always sorted from the largest (bottom) to the smallest (top).

We might use an array for each column representing the current geometry, for example:
'''

columns: dict[Column, list[int]] = {
    Column.ORIGIN: list(range(NUM_RINGS, 0, -1)),
    Column.MID: [],
    Column.DESTINATION: []
}

def print_columns(cols: dict[Column, list[int]] = columns, height: int = NUM_RINGS) -> None:
    print("\n")
    width = max(2 * height + 1, max(len(c.name) for c in Column))
    for level in range(height - 1, -1, -1):
        cells = []
        for col in Column:
            stack = cols[col]
            if level < len(stack):
                ring = stack[level]
                cells.append(str(ring).center(2 * ring - 1, "=").center(width))
            else:
                cells.append("|".center(width))
        print("  ".join(cells))
    print("  ".join("─" * width for _ in Column))
    print("  ".join(col.name.center(width) for col in Column))
    print("\n")
    
# Let's test an algo now
def move_top_ring(from_col: Column, to_col: Column, verbose: bool = False):
    '''
    Moves the top ring from `from_col` to `to_col`.
    Raises RuntimeError in the following cases:
    - No rings on the source column;
    - Attempting to stack onto a smaller ring
    '''
    if len(columns[from_col]) == 0:
        raise RuntimeError(f"Invalid move: no rings on the source column {from_col.name}")
    if len(columns[to_col]) > 0 and columns[from_col][-1] > columns[to_col][-1]:
        raise RuntimeError(f"Invalid move - attempting to stack ring {columns[from_col][-1]} onto {columns[to_col][-1]}")
    if verbose: print(f"Ring {columns[from_col][-1]} from column {from_col.name} to column {to_col.name}")
    columns[to_col].append(columns[from_col].pop())

def move_rings(number: int, from_col: Column, to_col: Column, verbose: bool = False):
    if number>1:
        buffer_col: Column = [x for x in Column if x not in [from_col, to_col]][0]
        move_rings(number-1, from_col, buffer_col, verbose)
        move_top_ring(from_col, to_col, verbose)
        move_rings(number-1, buffer_col, to_col, verbose)
    else:
        move_top_ring(from_col, to_col, verbose)

print("INITIAL STATE:")
print_columns()
move_rings(NUM_RINGS, Column.ORIGIN, Column.DESTINATION, verbose=True)
print("\nFINAL STATE:")
print_columns()
if columns[Column.DESTINATION] == list(range(NUM_RINGS, 0, -1)):
    print("\n---------- SUCCESS!!! ----------")
else:
    raise RuntimeError("Solver error: solution not reached!")

INITIAL STATE:


     1            |            |     
    =2=           |            |     
   ==3==          |            |     
───────────  ───────────  ───────────
   ORIGIN        MID      DESTINATION


Ring 1 from column ORIGIN to column DESTINATION
Ring 2 from column ORIGIN to column MID
Ring 1 from column DESTINATION to column MID
Ring 3 from column ORIGIN to column DESTINATION
Ring 1 from column MID to column ORIGIN
Ring 2 from column MID to column DESTINATION
Ring 1 from column ORIGIN to column DESTINATION

FINAL STATE:


     |            |            1     
     |            |           =2=    
     |            |          ==3==   
───────────  ───────────  ───────────
   ORIGIN        MID      DESTINATION



---------- SUCCESS!!! ----------


Now that it works, let's rewrite it as a class.

In [ ]:
from enum import IntEnum

class Column(IntEnum):
    ORIGIN = 0
    MID = 1
    DESTINATION = 2

class TowerOfHanoi():
    def __init__(self, number_of_rings: int = 3):
        self.reset(number_of_rings)
        
    def __str__(self):
        lines: list[str] = []
        width = max(2 * self._num_rings + 1, max(len(c.name) for c in Column))
        for level in range(self._num_rings - 1, -1, -1):
            cells = []
            for col in Column:
                stack = self._columns[col]
                if level < len(stack):
                    ring = stack[level]
                    cells.append(str(ring).center(2 * ring - 1, "=").center(width))
                else:
                    cells.append("|".center(width))
            lines.append("  ".join(cells))
        lines.append("  ".join("─" * width for _ in Column))
        lines.append("  ".join(col.name.center(width) for col in Column))
        return "\n".join(lines)

    def solve(self, verbose: bool = False):
        if self._solved:
            if verbose: print("Already solved")
            return
        if verbose: print("\n------------------- STEPS -------------------")
        self._move_rings(self._num_rings, Column.ORIGIN, Column.DESTINATION, verbose)
        if self._columns[Column.DESTINATION] == list(range(self._num_rings, 0, -1)):
            self._solved = True
            if verbose: print("------------------ SOLVED!!! ----------------")
        else:
            raise RuntimeError("Solver error: solution not reached!")

    def reset(self, number_of_rings: int | None = None):
        self._solved = False
        if number_of_rings is not None:
            if number_of_rings < 1:
                raise ValueError("You need at least one ring to play, duh!!")
            self._num_rings: int = number_of_rings
        self._columns: dict[Column, list[int]] = {
            Column.ORIGIN: list(range(self._num_rings, 0, -1)),
            Column.MID: [],
            Column.DESTINATION: []
        }

    def _move_top_ring(self, from_col: Column, to_col: Column, verbose: bool = False):
        '''
        Moves the top ring from `from_col` to `to_col`.
        Raises RuntimeError in the following cases:
        - No rings on the source column;
        - Attempting to stack onto a smaller ring
        '''
        if len(self._columns[from_col]) == 0:
            raise RuntimeError(f"Invalid move: no rings on the source column {from_col.name}")
        if len(self._columns[to_col]) > 0 and self._columns[from_col][-1] > self._columns[to_col][-1]:
            raise RuntimeError(f"Invalid move - attempting to stack ring {self._columns[from_col][-1]} onto {self._columns[to_col][-1]}")
        if verbose: print(f"Ring {self._columns[from_col][-1]} from column {from_col.name} to column {to_col.name}")
        self._columns[to_col].append(self._columns[from_col].pop())
    
    def _move_rings(self, number: int, from_col: Column, to_col: Column, verbose: bool = False):
        if number>1:
            buffer_col: Column = [x for x in Column if x not in [from_col, to_col]][0]
            self._move_rings(number-1, from_col, buffer_col, verbose)
            self._move_top_ring(from_col, to_col, verbose)
            self._move_rings(number-1, buffer_col, to_col, verbose)
        else:
            self._move_top_ring(from_col, to_col, verbose)

def test():
    my_tower: TowerOfHanoi = TowerOfHanoi(4)
    print("INITIAL STATE:\n")
    print(my_tower)
    my_tower.solve(verbose = True)
    print("\nFINAL STATE:")
    print(my_tower)

test()

INITIAL STATE:

     1            |            |     
    =2=           |            |     
   ==3==          |            |     
  ===4===         |            |     
───────────  ───────────  ───────────
   ORIGIN        MID      DESTINATION

------------------- STEPS -------------------
Ring 1 from column ORIGIN to column MID
Ring 2 from column ORIGIN to column DESTINATION
Ring 1 from column MID to column DESTINATION
Ring 3 from column ORIGIN to column MID
Ring 1 from column DESTINATION to column ORIGIN
Ring 2 from column DESTINATION to column MID
Ring 1 from column ORIGIN to column MID
Ring 4 from column ORIGIN to column DESTINATION
Ring 1 from column MID to column DESTINATION
Ring 2 from column MID to column ORIGIN
Ring 1 from column DESTINATION to column ORIGIN
Ring 3 from column MID to column DESTINATION
Ring 1 from column ORIGIN to column MID
Ring 2 from column ORIGIN to column DESTINATION
Ring 1 from column MID to column DESTINATION
------------------ SOLVED!!! ---------------

Next step: the Rust version replaces these three lists with a single `u64`, one bit per ring per peg.